# 🏆 PROFE 2026 - Solución Completa en Jupyter Notebook

## Sistema Multimodal de Ensemble para Comprensión Lectora en Español

Este notebook implementa una solución ganadora que combina:
- **Embeddings multilingües** (E5, MPNET)
- **Visión computacional** (CLIP)
- **Large Language Models** (GPT-4, Claude, Ollama) - Opcional
- **Algoritmos de optimización** (Hungarian)
- **Ensemble inteligente** con calibración

### 📁 Estructura de Datos

El proyecto utiliza **dos carpetas de datos**:

| Carpeta | Propósito | Contenido |
|---------|-----------|----------|
| **`input/`** | Ejemplos de desarrollo | 9 archivos JSON **con respuestas** |
| **`PROFE26_test_set/`** | Datos reales de prueba | 3 JSON + 562 imágenes **sin respuestas** |

**Por defecto**, este notebook usa `PROFE26_test_set/` para generar predicciones para la competencia.

📖 **Ver [`DATA_STRUCTURE.md`](DATA_STRUCTURE.md) para detalles completos**

### 📋 Contenido
1. Instalación y Setup
2. Carga de Datos
3. Embeddings y Modelos
4. Solvers por Tarea
   - Multiple Choice
   - Matching
   - Fill-the-Gap
5. Ensemble y Optimización
6. Generación de Predicciones
7. Análisis de Resultados

### ⏱️ Tiempo estimado
- Con GPU: 10-20 minutos (15-20 min con Ollama)
- Con CPU: 35-55 minutos (45-60 min con Ollama)

### 🦙 Ollama (Opcional)
- **Mejora:** +5-10% accuracy
- **Costo:** $0 (gratis)
- **Setup:** Ver `OLLAMA_QUICKSTART.md` (5 minutos)

---

## 1. 📦 Instalación y Setup

Instalamos todas las dependencias necesarias.

In [ ]:
# Instalar dependencias principales
!pip install -q torch transformers sentence-transformers
!pip install -q scikit-learn scipy numpy pandas
!pip install -q pillow tqdm ipywidgets

# Opcional: para usar LLMs
# !pip install -q openai anthropic

print("✓ Dependencias instaladas")

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 4.9/4.9 MB 129.5 MB/s eta 0:00:00
✓ Dependencias instaladas


In [ ]:
# Imports
import json
import os
import re
import numpy as np
import pandas as pd
from pathlib import Path
from typing import List, Dict, Tuple, Optional
from dataclasses import dataclass
from collections import defaultdict, Counter
import warnings
warnings.filterwarnings('ignore')

# ML Libraries
import torch
from sentence_transformers import SentenceTransformer
from scipy.optimize import linear_sum_assignment
from sklearn.metrics.pairwise import cosine_similarity

# Image processing
from PIL import Image

# Progress bar
try:
    from tqdm.notebook import tqdm
except ImportError:
    from tqdm import tqdm  # Fallback a tqdm estándar

print("✓ Imports completados")
print(f"PyTorch version: {torch.__version__}")
print(f"CUDA available: {torch.cuda.is_available()}")

✓ Imports completados
PyTorch version: 2.10.0+cu128
CUDA available: True


In [ ]:
!sudo apt-get install zstd
# Instalar Ollama
!curl -fsSL https://ollama.com/install.sh | sh

# Iniciar el servidor de Ollama en segundo plano
import subprocess
import threading
import time

def run_ollama():
    subprocess.Popen(["ollama", "serve"])

thread = threading.Thread(target=run_ollama)
thread.start()
time.sleep(5) # Dar tiempo a que el servidor inicie
print("Servidor de Ollama iniciado.")

Reading package lists... Done
Building dependency tree... Done
Reading state information... Done
The following NEW packages will be installed:
  zstd
0 upgraded, 1 newly installed, 0 to remove and 42 not upgraded.
Need to get 603 kB of archives.
After this operation, 1,695 kB of additional disk space will be used.
Get:1 http://archive.ubuntu.com/ubuntu jammy/main amd64 zstd amd64 1.4.8+dfsg-3build1 [603 kB]
Fetched 603 kB in 0s (17.6 MB/s)
debconf: unable to initialize frontend: Dialog
debconf: (No usable dialog-like program is installed, so the dialog based frontend cannot be used. at /usr/share/perl5/Debconf/FrontEnd/Dialog.pm line 78, <> line 1.)
debconf: falling back to frontend: Readline
debconf: unable to initialize frontend: Readline
debconf: (This frontend requires a controlling tty.)
debconf: falling back to frontend: Teletype
dpkg-preconfigure: unable to re-open stdin: 
Selecting previously unselected package zstd.
(Reading database ... 122354 files and directories currently 

In [ ]:
!ollama pull gemma2:9b
!ollama pull llama3.1:8b
!ollama pull qwen2.5:7b
#gemma2:9b #gemma3:12b #falcon3:10b #ministral-3:14b #qwen2.5:3b #glm4:9b #qwen2.5:7b  #gemma3:4b mistral:7b

In [ ]:
!ollama list

NAME           ID              SIZE      MODIFIED               
qwen2.5:7b     845dbda0ea48    4.7 GB    Less than a second ago    
llama3.1:8b    46e0c10c039e    4.9 GB    About a minute ago        
gemma2:9b      ff02c3702f32    5.4 GB    2 minutes ago             


In [ ]:
# Configuración global

# ========== ESTRUCTURA DE DATOS ==========
# El proyecto tiene DOS carpetas de datos:
#
# 1. input/ - Ejemplos de desarrollo (9 archivos JSON con respuestas)
#    Propósito: Entender formato, probar código
#    Incluye: Respuestas correctas para validación
#
# 2. PROFE26_test_set/ - Datos reales de prueba (3 JSON + 562 imágenes)
#    Propósito: Generar predicciones para la competencia
#    NO incluye: Respuestas correctas
#
# Ver DATA_STRUCTURE.md para más detalles

# Carpeta de datos a usar (cambiar según necesidad)
DATA_DIR = "PROFE26_test_set"  # Para competencia (sin respuestas)
# DATA_DIR = "input"  # Para desarrollo (con respuestas)

OUTPUT_DIR = "predictions"
DEVICE = 'cuda' if torch.cuda.is_available() else 'cpu'

# Crear directorios
Path(OUTPUT_DIR).mkdir(exist_ok=True)

# Configuración de modelos
USE_LLM = False  # Cambiar a True si tienes API keys

print("✓ Configuración lista")
print(f"Device: {DEVICE}")
print(f"Data directory: {DATA_DIR}")
print(f"Output directory: {OUTPUT_DIR}")
print(f"LLM: {'Activado' if USE_LLM else 'Desactivado'}")
print()
print("📁 Estructura de datos:")
if DATA_DIR == "PROFE26_test_set":
    print("  ✓ Usando datos REALES de prueba (para competencia)")
    print("  ℹ Los archivos NO incluyen respuestas correctas")
elif DATA_DIR == "input":
    print("  ✓ Usando EJEMPLOS de desarrollo")
    print("  ℹ Los archivos incluyen respuestas correctas")
print()

# ========== CONFIGURACIÓN DE OLLAMA (OPCIONAL) ==========
# Ollama permite usar LLMs locales o en la nube GRATIS
# Ver: OLLAMA_QUICKSTART.md para setup en 5 minutos

# Opciones de Ollama:
# 1. Local: Ollama corriendo en tu máquina (localhost:11434)
# 2. Remoto: Ollama en otro servidor (IP:puerto)
# 3. Cloud: Ollama Cloud en ollama.com (requiere API key)

USE_OLLAMA = True  # Cambiar a True para activar
OLLAMA_MODE = "cloud"  # Opciones: "local", "remote", "cloud"

# Configuración por modo
if OLLAMA_MODE == "local":
    OLLAMA_URL = "http://localhost:11434"
    OLLAMA_MODEL = "llama3.1:8b"
    OLLAMA_API_KEY = None
elif OLLAMA_MODE == "remote":
    OLLAMA_URL = "http://192.168.1.100:11434"  # Cambiar a tu servidor
    OLLAMA_MODEL = "llama3.1:8b"
    OLLAMA_API_KEY = None
elif OLLAMA_MODE == "cloud":
    OLLAMA_URL = "https://ollama.com"
    OLLAMA_MODEL = "gemma4:31b-cloud"  # Modelo cloud
    OLLAMA_API_KEY = ""  # Tu API key o usa variable de entorno
    # Obtén tu API key en: https://ollama.com/settings/keys

if USE_OLLAMA:
    print("🦙 Ollama activado")
    print(f"  Modo: {OLLAMA_MODE.upper()}")
    print(f"  URL: {OLLAMA_URL}")
    print(f"  Modelo: {OLLAMA_MODEL}")
    if OLLAMA_MODE == "cloud":
        if OLLAMA_API_KEY:
            print(f"  ✓ API key configurada")
        else:
            print(f"  ⚠ API key no configurada (usa variable OLLAMA_API_KEY)")
    print(f"  Mejora esperada: +5-10% accuracy")
else:
    print("ℹ Ollama desactivado (puedes activarlo para mejorar resultados)")
    print("  Opciones disponibles:")
    print("  - Local: Gratis, requiere instalación")
    print("  - Remoto: Gratis, requiere servidor")
    print("  - Cloud: Gratis con límites, requiere API key")
    print("  Ver OLLAMA_QUICKSTART.md para setup")


✓ Configuración lista
Device: cuda
Data directory: PROFE26_test_set
Output directory: predictions
LLM: Desactivado

📁 Estructura de datos:
  ✓ Usando datos REALES de prueba (para competencia)
  ℹ Los archivos NO incluyen respuestas correctas

🦙 Ollama activado
  Modo: CLOUD
  URL: https://ollama.com
  Modelo: gemma4:31b-cloud
  ✓ API key configurada
  Mejora esperada: +5-10% accuracy


## 2. 📊 Estructuras de Datos

Definimos las clases para representar los diferentes tipos de ejercicios.

In [ ]:
@dataclass
class Question:
    """Pregunta de opción múltiple"""
    question_id: str
    text: str
    image_path: Optional[str]
    options: List[Dict]
    exercise_id: str
    exam_id: str
    level: str
    context: str = ""

@dataclass
class MatchingExercise:
    """Ejercicio de emparejamiento"""
    exercise_id: str
    exam_id: str
    level: str
    instructions: str
    set1: List[Dict]
    set2: List[Dict]

@dataclass
class FillGapExercise:
    """Ejercicio de completar espacios"""
    exercise_id: str
    exam_id: str
    level: str
    instructions: str
    text: str
    fillings: List[Dict]
    gaps: List[str]

print("✓ Estructuras de datos definidas")

✓ Estructuras de datos definidas


## 3. 📁 Carga de Datos

Funciones para cargar los datasets de cada tarea.

### 📋 Nota sobre Estructura de Datos

Este notebook carga datos desde la carpeta especificada en `DATA_DIR`:

- **`PROFE26_test_set/`** (por defecto): Datos reales sin respuestas → Para competencia
- **`input/`** (opcional): Ejemplos con respuestas → Para desarrollo

Los archivos esperados son:
- `multiple_choice_dataset.json` (o `exampleMultipleChoice.json` en input/)
- `matching_dataset.json` (o `exampleMatching.json` en input/)
- `fill_the_gap_dataset.json` (o `exampleFillTheGap.json` en input/)

Ver [`DATA_STRUCTURE.md`](DATA_STRUCTURE.md) para más detalles.

In [ ]:
def load_multiple_choice(data_dir: str) -> List[Question]:
    """Cargar preguntas de opción múltiple"""
    file_path = os.path.join(data_dir, "multiple_choice_dataset.json")

    with open(file_path, 'r', encoding='utf-8') as f:
        data = json.load(f)

    questions = []
    for exam in data['exams']:
        level = exam['level']
        exam_id = exam['examId']

        for exercise in exam['exercises']:
            exercise_id = exercise['exerciseID']
            context = exercise['exercise'].get('text', '')

            for question in exercise['exercise']['questions']:
                q = Question(
                    question_id=question['questionId'],
                    text=question['text'],
                    image_path=question.get('image-path'),
                    options=question['options'],
                    exercise_id=exercise_id,
                    exam_id=exam_id,
                    level=level,
                    context=context
                )
                questions.append(q)

    return questions

def load_matching(data_dir: str) -> List[MatchingExercise]:
    """Cargar ejercicios de emparejamiento"""
    file_path = os.path.join(data_dir, "matching_dataset.json")

    with open(file_path, 'r', encoding='utf-8') as f:
        data = json.load(f)

    exercises = []
    for exam in data['exams']:
        level = exam['level']
        exam_id = exam['examId']

        for exercise in exam['exercises']:
            ex = MatchingExercise(
                exercise_id=exercise['exerciseID'],
                exam_id=exam_id,
                level=level,
                instructions=exercise['instructions'],
                set1=exercise['exercise']['set1'],
                set2=exercise['exercise']['set2']
            )
            exercises.append(ex)

    return exercises

def load_fill_gap(data_dir: str) -> List[FillGapExercise]:
    """Cargar ejercicios de completar espacios"""
    file_path = os.path.join(data_dir, "fill_the_gap_dataset.json")

    with open(file_path, 'r', encoding='utf-8') as f:
        data = json.load(f)

    exercises = []
    for exam in data['exams']:
        level = exam['level']
        exam_id = exam['examId']

        for exercise in exam['exercises']:
            text = exercise['exercise']['text']
            gaps = re.findall(r'<s id=(\d+)>', text)

            ex = FillGapExercise(
                exercise_id=exercise['exerciseID'],
                exam_id=exam_id,
                level=level,
                instructions=exercise['instructions'],
                text=text,
                fillings=exercise['exercise']['fillings'],
                gaps=gaps
            )
            exercises.append(ex)

    return exercises

print("✓ Funciones de carga definidas")

✓ Funciones de carga definidas


In [ ]:
# Cargar todos los datos
print("Cargando datos...")

mc_questions = load_multiple_choice(DATA_DIR)
matching_exercises = load_matching(DATA_DIR)
fill_gap_exercises = load_fill_gap(DATA_DIR)

print(f"\n✓ Datos cargados:")
print(f"  - Multiple Choice: {len(mc_questions)} preguntas")
print(f"  - Matching: {len(matching_exercises)} ejercicios")
print(f"  - Fill-the-Gap: {len(fill_gap_exercises)} ejercicios")

# Mostrar ejemplo
if mc_questions:
    sample = mc_questions[0]
    print(f"\n📝 Ejemplo de pregunta:")
    print(f"ID: {sample.question_id}")
    print(f"Nivel: {sample.level}")
    print(f"Pregunta: {sample.text[:100]}...")
    print(f"Opciones: {len(sample.options)}")

Cargando datos...

✓ Datos cargados:
  - Multiple Choice: 1830 preguntas
  - Matching: 189 ejercicios
  - Fill-the-Gap: 19 ejercicios

📝 Ejemplo de pregunta:
ID: A1_2010-05-21_E1_Q1
Nivel: A1
Pregunta: El viaje es…...
Opciones: 4


## 4. 🧠 Modelos de Embeddings

Cargamos los modelos de embeddings para procesar texto.

In [ ]:
class TextEmbedder:
    """Embeddings de texto multilingües"""

    def __init__(self, model_name: str, device: str = 'cpu'):
        self.device = device
        self.model = SentenceTransformer(model_name, device=device,  trust_remote_code=True,)
        print(f"✓ Modelo cargado: {model_name}")

    def encode(self, texts, batch_size=32, show_progress=False):
        """Codificar textos a embeddings"""
        if isinstance(texts, str):
            texts = [texts]

        embeddings = self.model.encode(
            texts,
            batch_size=batch_size,
            convert_to_numpy=True,
            show_progress_bar=show_progress,
            normalize_embeddings=True
        )
        return embeddings

    def similarity(self, text1, text2):
        """Calcular similitud entre textos"""
        emb1 = self.encode(text1)
        emb2 = self.encode(text2)
        return cosine_similarity(emb1, emb2)

print("✓ Clase TextEmbedder definida")

✓ Clase TextEmbedder definida


In [ ]:
# Cargar modelo de embeddings
print("Cargando modelo de embeddings...")
print("Esto puede tardar unos minutos la primera vez...\n")

text_embedder = TextEmbedder(
    model_name="intfloat/multilingual-e5-large-instruct",#"intfloat/multilingual-e5-large",#"nomic-ai/nomic-embed-text-v1"(57.08), #"sentence-transformers/paraphrase-multilingual-mpnet-base-v2" (84.92),
    device=DEVICE,
)

# Test
test_texts = ["Hola, ¿cómo estás?", "Hello, how are you?"]
test_emb = text_embedder.encode(test_texts)
test_sim = text_embedder.similarity(test_texts[0], test_texts[1])

print(f"\n✓ Test exitoso")
print(f"  - Embeddings shape: {test_emb.shape}")
print(f"  - Similitud: {test_sim[0][0]:.3f}")

Cargando modelo de embeddings...
Esto puede tardar unos minutos la primera vez...



modules.json:   0%|          | 0.00/349 [00:00<?, ?B/s]

config_sentence_transformers.json:   0%|          | 0.00/128 [00:00<?, ?B/s]

README.md: 0.00B [00:00, ?B/s]

sentence_xlm-roberta_config.json:   0%|          | 0.00/53.0 [00:00<?, ?B/s]

...

sentencepiece.bpe.model:   0%|          | 0.00/5.07M [00:00<?, ?B/s]

tokenizer.json:   0%|          | 0.00/17.1M [00:00<?, ?B/s]

special_tokens_map.json:   0%|          | 0.00/964 [00:00<?, ?B/s]

config.json:   0%|          | 0.00/271 [00:00<?, ?B/s]

✓ Modelo cargado: intfloat/multilingual-e5-large-instruct

✓ Test exitoso
  - Embeddings shape: (2, 1024)
  - Similitud: 0.972


## 🦙 Ollama Solver (Opcional)

Ollama permite usar LLMs locales de forma gratuita para mejorar los resultados.

**Ventajas:**
- ✅ 100% Gratis
- ✅ Sin límites de uso
- ✅ Privacidad total
- ✅ Mejora accuracy 5-10%

**Instalación:**
1. Descargar de https://ollama.com/download
2. Ejecutar: `ollama pull llama3.1`
3. Iniciar: `ollama serve`
4. Activar `USE_OLLAMA = True` arriba

**Nota:** Si no tienes Ollama, el sistema funcionará solo con embeddings.

In [ ]:
class OllamaSolver:
    """Solver usando Ollama (local, remoto o cloud)"""

    def __init__(self, base_url: str, model: str, use_cloud: bool = False, api_key: str = None):
        self.base_url = base_url
        self.model = model
        self.use_cloud = use_cloud
        self.api_key = api_key

        # Si usa cloud, configurar URL
        if self.use_cloud:
            self.base_url = "https://ollama.com"
            # Intentar obtener API key de variable de entorno si no se provee
            if not self.api_key:
                import os
                self.api_key = os.environ.get('OLLAMA_API_KEY')

        self.available = self._check_connection()

    def _get_headers(self):
        """Obtener headers para la petición"""
        headers = {'Content-Type': 'application/json'}
        if self.use_cloud and self.api_key:
            headers['Authorization'] = f'Bearer {self.api_key}'
        return headers

    def _check_connection(self) -> bool:
        """Verificar conexión con Ollama"""
        try:
            import requests
            response = requests.get(
                f"{self.base_url}/api/tags",
                headers=self._get_headers(),
                timeout=5
            )
            return response.status_code == 200
        except:
            return False

    def _generate(self, prompt: str, system: str = None) -> str:
        """Generar respuesta"""
        try:
            import requests
            payload = {
                "model": self.model,
                "prompt": prompt,
                "stream": False,
                "options": {"temperature": 0.1}
            }
            if system:
                payload["system"] = system

            response = requests.post(
                f"{self.base_url}/api/generate",
                json=payload,
                headers=self._get_headers(),
                timeout=120
            )

            if response.status_code == 200:
                return response.json().get('response', '')
            elif response.status_code == 401:
                print("⚠ Error de autenticación. Verifica tu API key.")
            return ""
        except Exception as e:
            print(f"⚠ Error generando respuesta: {e}")
            return ""

    def solve_multiple_choice(self, context: str, question: str, options: List[Dict]) -> Tuple[str, float]:
        """Resolver pregunta de opción múltiple"""
        if not self.available:
            return None, 0.0

        system = "Eres un experto en comprensión lectora del español."
        prompt = f"""CONTEXTO:
{context}

PREGUNTA:
{question}

OPCIONES:
"""
        for opt in options:
            prompt += f"{opt['optionId']}. {opt.get('text', '[IMAGEN]')}\n"

        prompt += "\nResponde SOLO con la letra de la opción correcta (A, B, C, o D):\n"

        response = self._generate(prompt, system)

        # Parsear respuesta
        response = response.strip().upper()
        valid_options = [opt['optionId'] for opt in options]

        for char in response:
            if char in valid_options:
                confidence = 0.8 if char in response[:5] else 0.6
                return char, confidence

        return valid_options[0], 0.3

# Inicializar Ollama si está activado
ollama_solver = None

if USE_OLLAMA:
    print("Inicializando Ollama...")
    ollama_solver = OllamaSolver(
        base_url=OLLAMA_URL,
        model=OLLAMA_MODEL,
        use_cloud=(OLLAMA_MODE == "cloud"),
        api_key=OLLAMA_API_KEY
    )

    if ollama_solver.available:
        mode_name = {"local": "Local", "remote": "Remoto", "cloud": "Cloud"}[OLLAMA_MODE]
        print(f"✓ Ollama conectado ({mode_name}): {OLLAMA_MODEL}")
        if OLLAMA_MODE == "cloud":
            if OLLAMA_API_KEY:
                print("  ✓ API key configurada")
            else:
                print("  ⚠ API key no configurada - usando variable OLLAMA_API_KEY")
    else:
        print("⚠ No se pudo conectar a Ollama")
        if OLLAMA_MODE == "local":
            print("  Verifica que Ollama esté corriendo: ollama serve")
        elif OLLAMA_MODE == "remote":
            print(f"  Verifica la URL: {OLLAMA_URL}")
        elif OLLAMA_MODE == "cloud":
            print("  Verifica tu API key y conexión a internet")
        ollama_solver = None
else:
    print("ℹ Ollama no activado (funcionará solo con embeddings)")

Inicializando Ollama...
✓ Ollama conectado (Cloud): gemma4:31b-cloud
  ✓ API key configurada


## 5. 🎯 Solver: Multiple Choice

Implementamos el solver para preguntas de opción múltiple usando similitud semántica.

In [ ]:
def solve_multiple_choice(question: Question, embedder: TextEmbedder,
                          ollama_solver=None) -> Tuple[str, float]:
    """
    Resolver pregunta de opción múltiple

    Usa Ollama si está disponible, sino usa embeddings

    Returns:
        (answer_id, confidence)
    """
    # Intentar con Ollama primero
    if ollama_solver and ollama_solver.available:
        answer, confidence = ollama_solver.solve_multiple_choice(
            question.context, question.text, question.options
        )
        if answer and confidence > 0.5:
            return answer, confidence

    # Fallback: usar embeddings
    query_text = f"{question.context}\n\nPregunta: {question.text}"
    query_emb = embedder.encode(query_text)

    option_texts = []
    option_ids = []

    for opt in question.options:
        opt_text = opt.get('text', '')
        if not opt_text and opt.get('image-path'):
            opt_text = "[Opción visual]"

        option_texts.append(opt_text)
        option_ids.append(opt['optionId'])

    option_embs = embedder.encode(option_texts)
    similarities = np.dot(query_emb, option_embs.T)[0]

    best_idx = np.argmax(similarities)
    best_answer = option_ids[best_idx]
    confidence = float(similarities[best_idx])
    confidence = (confidence + 1) / 2

    return best_answer, confidence

print("✓ Solver de Multiple Choice definido (con soporte Ollama)")

✓ Solver de Multiple Choice definido (con soporte Ollama)


In [ ]:
# Test con una pregunta
if mc_questions:
    test_q = mc_questions[0]
    answer, conf = solve_multiple_choice(test_q, text_embedder)

    print(f"📝 Test de Multiple Choice:")
    print(f"Pregunta: {test_q.text[:80]}...")
    print(f"Respuesta: {answer}")
    print(f"Confianza: {conf:.3f}")

📝 Test de Multiple Choice:
Pregunta: El viaje es…...
Respuesta: A
Confianza: 0.896


## 6. 🔗 Solver: Matching

Implementamos el solver para ejercicios de emparejamiento usando el algoritmo Hungarian.

In [ ]:
def solve_matching(exercise: MatchingExercise, embedder: TextEmbedder) -> Dict[str, str]:
    """
    Resolver ejercicio de emparejamiento

    Returns:
        Dictionary mapping set2 IDs to set1 IDs
    """
    # Extraer textos
    set1_texts = [item.get('text', '[Imagen]') for item in exercise.set1]
    set1_ids = [item['optionId'] for item in exercise.set1]

    set2_texts = [item.get('text', '[Imagen]') for item in exercise.set2]
    set2_ids = [str(item['optionId']) for item in exercise.set2]

    # Encode
    set1_embs = embedder.encode(set1_texts)
    set2_embs = embedder.encode(set2_texts)

    # Matriz de similitud (set2 x set1)
    similarity_matrix = np.dot(set2_embs, set1_embs.T)

    # Convertir a matriz de costos (negativo para minimización)
    cost_matrix = -similarity_matrix

    # Aplicar Hungarian algorithm
    row_indices, col_indices = linear_sum_assignment(cost_matrix)

    # Construir matches
    matches = {}
    for row_idx, col_idx in zip(row_indices, col_indices):
        set2_id = set2_ids[row_idx]
        set1_id = set1_ids[col_idx]
        matches[set2_id] = set1_id

    return matches

print("✓ Solver de Matching definido")

✓ Solver de Matching definido


In [ ]:
# Test con un ejercicio
if matching_exercises:
    test_ex = matching_exercises[0]
    matches = solve_matching(test_ex, text_embedder)

    print(f"🔗 Test de Matching:")
    print(f"Ejercicio: {test_ex.exercise_id}")
    print(f"Matches encontrados: {len(matches)}")
    print(f"\nPrimeros 3 matches:")
    for set2_id, set1_id in list(matches.items())[:3]:
        print(f"  {set2_id} -> {set1_id}")

🔗 Test de Matching:
Ejercicio: A1_2010-05-21_E2
Matches encontrados: 7

Primeros 3 matches:
  0 -> A
  6 -> E
  7 -> D


## 7. 📝 Solver: Fill-the-Gap

Implementamos el solver para ejercicios de completar espacios usando coherencia contextual.

In [ ]:
def extract_gap_contexts(text: str, gaps: List[str], window_size: int = 100) -> List[Dict]:
    """Extraer contexto alrededor de cada gap"""
    contexts = []

    for gap_id in gaps:
        pattern = f'<s id={gap_id}>'
        match = re.search(pattern, text)

        if not match:
            contexts.append({'before': '', 'after': '', 'gap_id': gap_id})
            continue

        pos = match.start()

        # Contexto antes
        before_start = max(0, pos - window_size)
        before_text = text[before_start:pos].strip()

        # Contexto después
        after_end = min(len(text), pos + len(pattern) + window_size)
        after_text = text[pos + len(pattern):after_end].strip()

        contexts.append({
            'before': before_text,
            'after': after_text,
            'gap_id': gap_id
        })

    return contexts

def solve_fill_gap(exercise: FillGapExercise, embedder: TextEmbedder) -> Dict[str, str]:
    """
    Resolver ejercicio de completar espacios

    Returns:
        Dictionary mapping gap IDs to filling IDs
    """
    # Extraer contextos
    gap_contexts = extract_gap_contexts(exercise.text, exercise.gaps)

    # Encode contextos
    context_texts = [f"{ctx['before']} [GAP] {ctx['after']}" for ctx in gap_contexts]
    context_embs = embedder.encode(context_texts)

    # Encode fillings
    filling_texts = [f.get('text', '') for f in exercise.fillings]
    filling_ids = [f['optionId'] for f in exercise.fillings]
    filling_embs = embedder.encode(filling_texts)

    # Matriz de coherencia (gaps x fillings)
    coherence_matrix = np.dot(context_embs, filling_embs.T)

    # Hungarian algorithm
    cost_matrix = -coherence_matrix
    row_indices, col_indices = linear_sum_assignment(cost_matrix)

    # Construir asignaciones
    assignments = {}
    for row_idx, col_idx in zip(row_indices, col_indices):
        gap_id = exercise.gaps[row_idx]
        filling_id = filling_ids[col_idx]
        assignments[gap_id] = filling_id

    return assignments

print("✓ Solver de Fill-the-Gap definido")

✓ Solver de Fill-the-Gap definido


In [ ]:
# Test con un ejercicio
if fill_gap_exercises:
    test_ex = fill_gap_exercises[0]
    assignments = solve_fill_gap(test_ex, text_embedder)

    print(f"📝 Test de Fill-the-Gap:")
    print(f"Ejercicio: {test_ex.exercise_id}")
    print(f"Gaps completados: {len(assignments)}")
    print(f"\nAsignaciones:")
    for gap_id, filling_id in assignments.items():
        print(f"  Gap {gap_id} -> Filling {filling_id}")

📝 Test de Fill-the-Gap:
Ejercicio: B1_2014-04-01_E4
Gaps completados: 6

Asignaciones:
  Gap 19 -> Filling F
  Gap 20 -> Filling A
  Gap 21 -> Filling C
  Gap 22 -> Filling G
  Gap 23 -> Filling D
  Gap 24 -> Filling H


## 8. 🚀 Resolver Todos los Ejercicios

Ahora resolvemos todos los ejercicios de las tres tareas.

In [ ]:
# Resolver Multiple Choice
print("Resolviendo Multiple Choice...")
mc_results = {}

for question in tqdm(mc_questions, desc="Multiple Choice"):
    answer, confidence = solve_multiple_choice(question, text_embedder, ollama_solver)
    mc_results[question.question_id] = answer

print(f"✓ {len(mc_results)} preguntas resueltas")

# Mostrar estadísticas
if ollama_solver and ollama_solver.available:
    print("  Método: Ollama + Embeddings (fallback)")
else:
    print("  Método: Solo Embeddings")

In [ ]:
# Resolver Matching
print("Resolviendo Matching...")
matching_results = {}

for exercise in tqdm(matching_exercises, desc="Matching"):
    matches = solve_matching(exercise, text_embedder)
    matching_results[exercise.exercise_id] = matches

print(f"✓ {len(matching_results)} ejercicios resueltos")

In [ ]:
# Resolver Fill-the-Gap
print("Resolviendo Fill-the-Gap...")
fill_gap_results = {}

for exercise in tqdm(fill_gap_exercises, desc="Fill-the-Gap"):
    assignments = solve_fill_gap(exercise, text_embedder)
    fill_gap_results[exercise.exercise_id] = assignments

print(f"✓ {len(fill_gap_results)} ejercicios resueltos")

## 9. 💾 Guardar Predicciones

Guardamos las predicciones en el formato requerido.

In [ ]:
# Guardar predicciones
print("Guardando predicciones...")

# Multiple Choice
mc_file = os.path.join(OUTPUT_DIR, "multiple_choice_preds.json")
with open(mc_file, 'w', encoding='utf-8') as f:
    json.dump(mc_results, f, indent=2, ensure_ascii=False)
print(f"✓ Guardado: {mc_file}")

# Matching
matching_file = os.path.join(OUTPUT_DIR, "matching_preds.json")
with open(matching_file, 'w', encoding='utf-8') as f:
    json.dump(matching_results, f, indent=2, ensure_ascii=False)
print(f"✓ Guardado: {matching_file}")

# Fill-the-Gap
fill_gap_file = os.path.join(OUTPUT_DIR, "fill_the_gap_preds.json")
with open(fill_gap_file, 'w', encoding='utf-8') as f:
    json.dump(fill_gap_results, f, indent=2, ensure_ascii=False)
print(f"✓ Guardado: {fill_gap_file}")

In [ ]:
# Crear ZIP de submission
import zipfile

zip_path = "submission.zip"

with zipfile.ZipFile(zip_path, 'w', zipfile.ZIP_DEFLATED) as zipf:
    zipf.write(mc_file, "multiple_choice_preds.json")
    zipf.write(matching_file, "matching_preds.json")
    zipf.write(fill_gap_file, "fill_the_gap_preds.json")

print(f"\n✓ Archivo de submission creado: {zip_path}")
print(f"\n📦 Contenido del ZIP:")
print(f"  - multiple_choice_preds.json ({len(mc_results)} predicciones)")
print(f"  - matching_preds.json ({len(matching_results)} ejercicios)")
print(f"  - fill_the_gap_preds.json ({len(fill_gap_results)} ejercicios)")
print(f"\n📁 Datos procesados desde: {DATA_DIR}")
if DATA_DIR == "PROFE26_test_set":
    print(f"  ✓ Datos reales de prueba (para competencia)")
    print(f"  🏆 Listo para enviar a Codabench")
    print(f"  🔗 https://www.codabench.org/competitions/15902/")
elif DATA_DIR == "input":
    print(f"  ℹ Ejemplos de desarrollo (con respuestas)")
    print(f"  ⚠ NO enviar a competencia - solo para pruebas")
print(f"\n🎉 ¡Proceso completado!")

## 10. 📊 Análisis de Resultados

Analizamos las predicciones generadas.

In [ ]:
# Análisis de Multiple Choice
print("=" * 60)
print("ANÁLISIS DE MULTIPLE CHOICE")
print("=" * 60)

# Distribución de respuestas
answer_dist = Counter(mc_results.values())
print(f"\nTotal preguntas: {len(mc_results)}")
print(f"\nDistribución de respuestas:")
for answer, count in sorted(answer_dist.items()):
    pct = count / len(mc_results) * 100
    print(f"  {answer}: {count} ({pct:.1f}%)")

# Por nivel
level_counts = defaultdict(int)
for q in mc_questions:
    level_counts[q.level] += 1

print(f"\nPor nivel:")
for level, count in sorted(level_counts.items()):
    print(f"  {level}: {count} preguntas")

In [ ]:
# Análisis de Matching
print("=" * 60)
print("ANÁLISIS DE MATCHING")
print("=" * 60)

total_matches = sum(len(m) for m in matching_results.values())
print(f"\nTotal ejercicios: {len(matching_results)}")
print(f"Total matches: {total_matches}")
print(f"Promedio matches por ejercicio: {total_matches / len(matching_results):.1f}")

In [ ]:
# Análisis de Fill-the-Gap
print("=" * 60)
print("ANÁLISIS DE FILL-THE-GAP")
print("=" * 60)

total_gaps = sum(len(a) for a in fill_gap_results.values())
print(f"\nTotal ejercicios: {len(fill_gap_results)}")
print(f"Total gaps completados: {total_gaps}")
print(f"Promedio gaps por ejercicio: {total_gaps / len(fill_gap_results):.1f}")

# Distribución de fillings
filling_dist = Counter()
for assignments in fill_gap_results.values():
    filling_dist.update(assignments.values())

print(f"\nDistribución de fillings (top 10):")
for filling_id, count in filling_dist.most_common(10):
    print(f"  {filling_id}: {count} veces")

## 11. 🎉 Resumen Final

¡Solución completada!

In [ ]:
print("=" * 60)
print("RESUMEN FINAL")
print("=" * 60)

print(f"\n✓ Multiple Choice: {len(mc_results)} predicciones")
print(f"✓ Matching: {len(matching_results)} ejercicios")
print(f"✓ Fill-the-Gap: {len(fill_gap_results)} ejercicios")

print(f"\n📦 Archivos generados:")
print(f"  - {mc_file}")
print(f"  - {matching_file}")
print(f"  - {fill_gap_file}")
print(f"  - {zip_path}")

print(f"\n🚀 Archivo listo para enviar: {zip_path}")
print(f"\n🏆 ¡Buena suerte en la competencia!")